# Modelo Copeland-Galai

Parametrizado por el dígito `d` a través de $\alpha = 0.d$.
Parámetros fijos: $S_0=100$, $\beta=0.05$, $\pi_I=0.30$, $\pi_L=0.70$,
y la densidad discreta $f(P)$ dada en clase.

Se resuelve para $d=2$ y $d=8$, y se verifica con $d=5$.

In [1]:
import numpy as np
from scipy.optimize import minimize

## Parámetros fijos del modelo

In [2]:
S0 = 100
beta = 0.05
piI = 0.30   # prob. trader informado
piL = 0.70   # prob. trader desinformado

# Distribución discreta de P (valor real del activo)
P_values = np.array([90, 95, 100, 105, 110])
f_P      = np.array([0.10, 0.20, 0.40, 0.20, 0.10])

## Funciones del modelo

In [3]:
def piLB(A, alpha):
    """Prob. de que el trader desinformado compre (ask=A)."""
    x = A - S0
    return max(alpha - beta * x, 0)


def piLS(B, alpha):
    """Prob. de que el trader desinformado venda (bid=B)."""
    x = S0 - B
    return max(alpha - beta * x, 0)


def G(A, B, alpha):
    """Ganancia esperada del dealer proveniente del trader desinformado."""
    return piL * (piLB(A, alpha) * (A - S0) + piLS(B, alpha) * (S0 - B))


def L(A, B):
    """Pérdida esperada del dealer frente al trader informado."""
    perdida_ask = np.sum(np.where(P_values > A, (P_values - A) * f_P, 0))
    perdida_bid = np.sum(np.where(P_values < B, (B - P_values) * f_P, 0))
    return piI * (perdida_ask + perdida_bid)


def profit(A, B, alpha):
    """Profit esperado total del dealer: pi(A,B) = G(A,B) - L(A,B)."""
    return G(A, B, alpha) - L(A, B)


def neg_profit(params, alpha):
    A, B = params
    return -profit(A, B, alpha)

## Optimización

$\pi_{LB}$ y $\pi_{LS}$ son $\max(\alpha-\beta x,0)$, así que la función
de profit tiene **mesetas planas**. Un solo arranque de un método de
gradiente (p. ej. L-BFGS-B) se puede quedar atrapado en un óptimo falso.
Por eso se hace primero una búsqueda en grilla y se refina el mejor
punto con Nelder-Mead.

In [4]:
def optimizar(alpha):
    """Encuentra (A*, B*) que maximizan el profit esperado del dealer,
    respetando Bid <= S0 <= Ask."""
    bounds = [(S0, 130), (70, S0)]  # A >= S0 ; B <= S0

    A_grid = np.linspace(S0, 130, 121)
    B_grid = np.linspace(70, S0, 121)
    mejor_profit = -np.inf
    mejor_AB = (S0, S0)
    for A in A_grid:
        for B in B_grid:
            p = profit(A, B, alpha)
            if p > mejor_profit:
                mejor_profit = p
                mejor_AB = (A, B)

    res = minimize(neg_profit, x0=mejor_AB, args=(alpha,),
                    method='Nelder-Mead',
                    bounds=bounds)
    A_opt, B_opt = res.x
    profit_opt = -res.fun
    if profit_opt < mejor_profit:
        A_opt, B_opt, profit_opt = mejor_AB[0], mejor_AB[1], mejor_profit
    return A_opt, B_opt, profit_opt


def reportar(nombre, alpha):
    A_opt, B_opt, profit_opt = optimizar(alpha)
    print(f"--- {nombre} (alpha = {alpha}) ---")
    print(f"A* (ask óptimo) = {A_opt:.4f}")
    print(f"B* (bid óptimo) = {B_opt:.4f}")
    print(f"G(A*,B*)        = {G(A_opt, B_opt, alpha):.4f}")
    print(f"L(A*,B*)        = {L(A_opt, B_opt):.4f}")
    print(f"Profit óptimo   = {profit_opt:.4f}")
    return A_opt, B_opt, profit_opt

## Verificación con el caso manual ($d=5$, $\alpha=0.5$, $A=105$, $B=95$)

In [5]:
alpha_5 = 0.5
print(f"G(105,95) = {G(105, 95, alpha_5):.4f}  (esperado 1.75)")
print(f"L(105,95) = {L(105, 95):.4f}  (esperado 0.30)")
print(f"Profit(105,95) = {profit(105, 95, alpha_5):.4f}  (esperado 1.45)")

G(105,95) = 1.7500  (esperado 1.75)
L(105,95) = 0.3000  (esperado 0.30)
Profit(105,95) = 1.4500  (esperado 1.45)


## Caso $d=2$

In [6]:
_ = reportar("d=2", 0.2)

--- d=2 (alpha = 0.2) ---
A* (ask óptimo) = 110.0000
B* (bid óptimo) = 70.0000
G(A*,B*)        = 0.0000
L(A*,B*)        = 0.0000
Profit óptimo   = 0.0000


**Interpretación:** con $\alpha$ tan bajo, cualquier spread que sí
atraiga operaciones genera más pérdida por selección adversa (trader
informado) que ganancia por el spread frente al trader desinformado.
El óptimo es un spread tan ancho que nadie transa
($A^*\ge 110$, $B^*\le 90$), con $G=L=\text{profit}=0$: solución de
esquina.

## Caso $d=8$ 

In [7]:
_ = reportar("d=8", 0.8)

--- d=8 (alpha = 0.8) ---
A* (ask óptimo) = 108.4286
B* (bid óptimo) = 91.5714
G(A*,B*)        = 4.4671
L(A*,B*)        = 0.0943
Profit óptimo   = 4.3729
